# OpenMed: three full-model fine-tuning runs

Select a CUDA GPU runtime. Run setup and preflight first. Training remains disabled until you set RUN_TRAINING=True in the marked cell. Each group keeps only its best model and Validation results. V4 is excluded.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%pip install -q --only-binary=sentencepiece transformers==4.51.3 tokenizers==0.21.1 sentencepiece==0.2.2 safetensors==0.5.3 "setuptools<82"

In [ ]:
from pathlib import Path
import json, sys, subprocess, os, hashlib
sys.dont_write_bytecode=True
p=Path('/content/drive/MyDrive/Holokai_Project/week9/OpenMed-NER-PharmaDetect-SuperClinical-434M')
helpers=json.loads(r'''"\ndef completed_results(project, hashes=None):\n    entries=[]\n    for path in (project/'outputs').glob('*/validation_results.json'):\n        result=json.loads(path.read_text())\n        if not result.get('completed'): continue\n        if hashes is not None: check(result['data_sha256']==hashes, f'Dataset mismatch: {path}')\n        entries.append(result)\n    def order(result):\n        name=result['group']\n        return (int(name[3:5]) if name.startswith('run') and name[3:5].isdigit() else 0,\n                result['configuration']['learning_rate'],name)\n    return sorted(entries,key=order)\n\ndef global_selection(project):\n    path=project/'outputs'/'best_model'/'selection.json'\n    return json.loads(path.read_text()) if path.is_file() else None\n\ndef consolidate_best(project, hashes):\n    import shutil\n    results=completed_results(project,hashes)\n    check(bool(results),'Existing completed results are required before adaptive tuning')\n    destination=project/'outputs'/'best_model'\n    if not destination.exists():\n        winner=max(results,key=lambda r:r['metrics']['overall']['f1'])\n        source=project/'outputs'/winner['group']/'best_model'\n        check((source/'model.safetensors').is_file() and (source/'config.json').is_file(),'Existing best checkpoint is incomplete')\n        source.rename(destination)\n    selected=global_selection(project)\n    check(selected is not None and selected['data_sha256']==hashes,'Overall checkpoint provenance mismatch')\n    best_completed=max(r['metrics']['overall']['f1'] for r in results)\n    check(selected['validation_micro_f1']>=best_completed,'Overall checkpoint is worse than preserved completed results')\n    # Retain every result JSON; only redundant model directories are removed.\n    for result in results:\n        redundant=project/'outputs'/result['group']/'best_model'\n        if redundant.is_dir() and not redundant.is_symlink(): shutil.rmtree(redundant)\n    print('Overall incumbent:',selected['group'],selected['validation_micro_f1'],flush=True)\n\ndef export_overall(model, tokenizer, mapping, selection, project):\n    import shutil\n    from safetensors import safe_open\n    root=project/'outputs'\n    destination=root/'best_model'\n    pending=root/'.best_model_pending'\n    previous=root/'.best_model_previous'\n    check(not pending.exists() and not previous.exists(),'Incomplete checkpoint promotion needs inspection; existing best is preserved')\n    model.save_pretrained(pending,safe_serialization=True)\n    tokenizer.save_pretrained(pending)\n    write_json(pending/'label_map.json',mapping)\n    write_json(pending/'selection.json',selection)\n    with safe_open(pending/'model.safetensors',framework='pt',device='cpu') as saved:\n        check(tuple(saved.get_slice('classifier.weight').get_shape())==(23,1024),'Exported classifier shape mismatch')\n    check(json.loads((pending/'config.json').read_text())['id2label']=={str(i):t for i,t in enumerate(TAGS)},'Exported label schema mismatch')\n    if destination.exists(): destination.rename(previous)\n    try: pending.rename(destination)\n    except Exception:\n        if previous.exists(): previous.rename(destination)\n        raise\n    if previous.exists(): shutil.rmtree(previous)\n    print('Saved overall best:',selection['group'],'epoch',selection['epoch'],selection['validation_micro_f1'],flush=True)\n\ndef propose_next(project, number, results):\n    best=max(results,key=lambda r:r['metrics']['overall']['f1'])\n    cfg=best['configuration']\n    keys={'name','learning_rate','epochs','seed','batch_size','gradient_accumulation_steps','weight_decay','warmup_ratio','max_grad_norm','gradient_checkpointing'}\n    proposal={k:cfg[k] for k in keys}\n    tried={(r['configuration']['learning_rate'],r['configuration']['epochs']) for r in results}\n    if best['best_epoch']>=cfg['epochs']-1 and cfg['epochs']<12 and (cfg['learning_rate'],12) not in tried:\n        proposal['epochs']=12\n        reason=f\"Best run {best['group']} peaked late ({best['best_epoch']}/{cfg['epochs']}); test a longer 12-epoch schedule at the same learning rate.\"\n    else:\n        rate=cfg['learning_rate']\n        rates=sorted({r['configuration']['learning_rate'] for r in results})\n        neighbors=sorted([x for x in rates if x<rate],reverse=True)[:1]+[x for x in rates if x>rate][:1]\n        neighbors.sort(key=lambda x:max(r['metrics']['overall']['f1'] for r in results if r['configuration']['learning_rate']==x),reverse=True)\n        candidates=[(rate+x)/2 for x in neighbors]+[rate*.8,rate*1.2]\n        proposal['epochs']=cfg['epochs'] if cfg['epochs']>5 else 8\n        for candidate in candidates:\n            candidate=float(f'{candidate:.8g}')\n            if 1e-5<=candidate<=6e-5 and (candidate,proposal['epochs']) not in tried:\n                proposal['learning_rate']=candidate\n                break\n        else: return None,'No distinct local learning-rate candidate remains.'\n        reason=f\"Refine learning rate around {best['group']} (Validation F1 {best['metrics']['overall']['f1']:.6f}); keep its epoch budget.\"\n    rate_name=f\"{proposal['learning_rate']/1e-5:.4g}\".replace('.','p')+'e-5'\n    proposal['name']=f\"run{number:02d}_lr{rate_name}_ep{proposal['epochs']}\"\n    return proposal,reason\n\ndef adaptive_runs(paths, project, data, mapping, manifest, hashes):\n    for path in paths:\n        config=read_config(path)\n        output=project/'outputs'/config['name']/'validation_results.json'\n        if output.is_file() and json.loads(output.read_text()).get('completed'):\n            print('Preserving completed run:',config['name'],flush=True)\n            continue\n        check(not output.parent.exists(),f'Incomplete output exists: {output.parent}; will not silently rerun')\n        run_group(config,project,data,mapping,manifest,hashes,reason='Initial extension: eight epochs; other settings match the previous runs.')\n    for number in range(6,9):\n        results=completed_results(project,hashes)\n        adaptive=[r for r in results if r['group'].startswith(('run06','run07','run08'))]\n        if any(r['group'].startswith(f'run{number:02d}_') for r in adaptive): continue\n        if len(adaptive)>=2 and all(r.get('global_improvement',0)<0.001 for r in adaptive[-2:]):\n            reason='Stopped: two successive adaptive runs improved overall Validation micro F1 by less than 0.001.'\n            last=adaptive[-1];last['adaptive_stop_reason']=reason\n            write_json(project/'outputs'/last['group']/'validation_results.json',last)\n            print(reason,flush=True);break\n        config,reason=propose_next(project,number,results)\n        if config is None: print(reason,flush=True);break\n        path=project/'configs'/f\"{config['name']}.json\"\n        check(not path.exists(),f'Configuration already exists without completed results: {path}')\n        write_json(path,config)\n        print('ADAPTIVE DECISION:',reason, json.dumps(config),flush=True)\n        run_group(config,project,data,mapping,manifest,hashes,reason=reason)\n    print('FINAL COMPARISON',flush=True)\n    for result in completed_results(project,hashes):\n        print(json.dumps({k:result[k] for k in ('group','best_epoch')}|{'learning_rate':result['configuration']['learning_rate'],'max_epochs':result['configuration']['epochs'],**result['metrics']['overall']}),flush=True)\n    print('FINAL BEST',json.dumps(global_selection(project)),flush=True)\n"''')
new_main=json.loads(r'''"\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('--project-dir',type=Path,default=PROJECT)\n    parser.add_argument('--configs',nargs='+',type=Path)\n    parser.add_argument('--adaptive',action='store_true',help='Run/skip completed Runs 4-5, then select at most Runs 6-8 using Validation only')\n    parser.add_argument('--check-data',action='store_true',help='Read-only checks, without model initialization')\n    args=parser.parse_args()\n    project=args.project_dir.resolve()\n    paths=args.configs or [project/'configs'/name for name in ('run04_lr3e-5_ep8.json','run05_lr4e-5_ep8.json')]\n    configs=[read_config(path) for path in paths]\n    check(len({c['name'] for c in configs})==len(configs),'Repeated configuration group')\n    data,mapping,manifest,summaries,hashes=load_prepared(project)\n    if args.check_data:\n        print(json.dumps(dict(status='PASS',configurations=configs,data=summaries),indent=2));return\n    if args.adaptive:\n        check([c['name'] for c in configs]==['run04_lr3e-5_ep8','run05_lr4e-5_ep8'],'Adaptive mode starts only from Runs 4-5')\n    else:\n        for c in configs: check(not (project/'outputs'/c['name']).exists(),f\"Refusing to rerun {c['name']}\")\n    consolidate_best(project,hashes)\n    if args.adaptive:\n        adaptive_runs(paths,project,data,mapping,manifest,hashes)\n    else:\n        for c in configs:run_group(c,project,data,mapping,manifest,hashes)\n        print('FINAL BEST',json.dumps(global_selection(project)),flush=True)\n\nif __name__=='__main__':\n    main()\n"''')
s=(p/'train.py').read_text()
assert 'def adaptive_runs(' not in s, 'Adaptive code already present; inspect before modifying'
s=s.replace('import math\n','import math\nimport time\n')
s=s.replace("check(config['name'] in GROUPS and path.stem==config['name'], 'Configuration/group name mismatch')","check(isinstance(config['name'],str) and path.stem==config['name'] and config['name'].replace('_','').replace('-','').replace('.','').isalnum(), 'Configuration/group name mismatch')")
s=s.replace('def run_group(config, project, data, mapping, manifest, hashes):',helpers+'\ndef run_group(config, project, data, mapping, manifest, hashes, reason=None):')
s=s.replace("    set_seed(config['seed'])","    started=time.monotonic()\n    incumbent=global_selection(project)\n    starting_global_score=incumbent['validation_micro_f1'] if incumbent else -1.0\n    set_seed(config['seed'])")
s=s.replace("    resolved['transformers_version']=transformers.__version__","    resolved['transformers_version']=transformers.__version__\n    resolved['tuning_reason']=reason or 'Explicit configuration run'")
old="""            destination=group_dir/'best_model'
            model.save_pretrained(destination,safe_serialization=True)
            tokenizer.save_pretrained(destination)
            write_json(destination/'label_map.json',mapping)
            write_json(destination/'selection.json',dict(group=config['name'],epoch=epoch,validation_micro_f1=score,configuration=resolved,data_sha256=hashes))
            write_json(group_dir/'validation_results.json',best_result)"""
new="""            best_result['global_improvement']=max(0.0,score-starting_global_score)
            selected=global_selection(project)
            if selected is None or better(score,selected['validation_micro_f1']):
                export_overall(model,tokenizer,mapping,dict(group=config['name'],epoch=epoch,validation_micro_f1=score,configuration=resolved,data_sha256=hashes),project)
            write_json(group_dir/'validation_results.json',best_result)"""
assert old in s
s=s.replace(old,new)
s=s.replace("    best_result['completed']=True","    best_result['completed']=True\n    best_result['duration_seconds']=round(time.monotonic()-started,2)")
s=s.replace('        metrics=evaluate_model',"            if (index+1)%300==0: print(f\"{config['name']} epoch {epoch}: batch {index+1}/{len(loader)}\",flush=True)\n        metrics=evaluate_model")
s=s[:s.index('def main():')]+new_main
compile(s,'train.py','exec')
base=json.loads((p/'configs/first_try/learning_rate_3e-5.json').read_text())
configs=[]
for name,lr in [('run04_lr3e-5_ep8',3e-5),('run05_lr4e-5_ep8',4e-5)]:
    config=dict(base,name=name,learning_rate=lr,epochs=8)
    path=p/'configs'/f'{name}.json'
    assert not path.exists(), f'Refusing overwrite: {path}'
    configs.append((path,config))
assert not (p/'finetune.ipynb').exists(), 'finetune.ipynb already exists; inspect before modifying'
(p/'train.py').write_text(s)
for path,config in configs:path.write_text(json.dumps(config,indent=2)+'\n')
sys.path.insert(0,str(p))
import train
for path,_ in configs:train.read_config(path)
fixture={'group':'run04_lr3e-5_ep8','configuration':dict(base,name='run04_lr3e-5_ep8',epochs=8),'best_epoch':8,'metrics':{'overall':{'f1':0.91}}}
proposal,why=train.propose_next(p,6,[fixture])
assert proposal['epochs']==12 and proposal['learning_rate']==3e-5
fixture['best_epoch']=4
other=dict(fixture,group='run05_lr4e-5_ep8',configuration=dict(fixture['configuration'],learning_rate=4e-5),metrics={'overall':{'f1':0.90}})
proposal,why=train.propose_next(p,6,[fixture,other])
assert abs(proposal['learning_rate']-3.5e-5)<1e-12
print('Adaptive policy synthetic tests: PASS')
result=subprocess.run([sys.executable,str(p/'train.py'),'--check-data'],env=dict(os.environ,PYTHONDONTWRITEBYTECODE='1'),capture_output=True,text=True)
print(result.stdout);print(result.stderr);assert result.returncode==0


In [ ]:
import json, nbformat, subprocess, sys, os
from pathlib import Path
p=Path('/content/drive/MyDrive/Holokai_Project/week9/OpenMed-NER-PharmaDetect-SuperClinical-434M')
nb=json.loads(r'''{"nbformat":4,"nbformat_minor":5,"metadata":{"colab":{"name":"finetune.ipynb","provenance":[]},"kernelspec":{"name":"python3","display_name":"Python 3"},"language_info":{"name":"python"},"accelerator":"GPU"},"cells":[{"cell_type":"markdown","id":"adaptive-0","metadata":{},"source":["# OpenMed 434M: adaptive tuning from Run 4\n","\n","The three completed first-try experiments are preserved and never rerun. Train/Validation only; V4 is excluded. Each run starts from the pinned original pretrained checkpoint. The script retains one overall best model plus concise results for each run. Adaptive mode tries Runs 4–5, then up to three Validation-guided experiments; it stops after two adaptive runs each improve overall micro F1 by less than 0.001."]},{"cell_type":"code","id":"adaptive-1","metadata":{},"source":["from google.colab import drive\n","drive.mount('/content/drive')\n"],"execution_count":null,"outputs":[]},{"cell_type":"code","id":"adaptive-2","metadata":{},"source":["%pip install -q --only-binary=sentencepiece transformers==4.51.3 tokenizers==0.21.1 sentencepiece==0.2.2 safetensors==0.5.3 \"setuptools<82\"\n"],"execution_count":null,"outputs":[]},{"cell_type":"code","id":"adaptive-3","metadata":{},"source":["from pathlib import Path\n","import os, subprocess, sys\n","PROJECT=Path('/content/drive/MyDrive/Holokai_Project/week9/OpenMed-NER-PharmaDetect-SuperClinical-434M')\n","CONFIGS=[PROJECT/'configs'/name for name in ('run04_lr3e-5_ep8.json','run05_lr4e-5_ep8.json')]\n","ADAPTIVE=True\n","ENV=dict(os.environ,PYTHONDONTWRITEBYTECODE='1')\n","subprocess.run([sys.executable,str(PROJECT/'evaluate.py'),'--self-test'],env=ENV,check=True)\n","subprocess.run([sys.executable,str(PROJECT/'train.py'),'--check-data','--configs',*map(str,CONFIGS)],env=ENV,check=True)\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"adaptive-4","metadata":{},"source":["## Execute experiments\n","\n","Set RUN_TRAINING=True to launch. For a later explicit config, change CONFIGS and set ADAPTIVE=False. Adaptive mode skips completed Run 4–8 results; incomplete runs are not silently restarted."]},{"cell_type":"code","id":"adaptive-5","metadata":{},"source":["RUN_TRAINING=False\n","if RUN_TRAINING:\n","    command=[sys.executable,'-u',str(PROJECT/'train.py'),'--configs',*map(str,CONFIGS)]+(['--adaptive'] if ADAPTIVE else [])\n","    with subprocess.Popen(command,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:\n","        for line in process.stdout: print(line,end='',flush=True)\n","        assert process.wait()==0, 'Training stopped; inspect the error above.'\n","else:\n","    print('Training disabled until RUN_TRAINING=True.')\n"],"execution_count":null,"outputs":[]},{"cell_type":"code","id":"adaptive-6","metadata":{},"source":["import json\n","import pandas as pd\n","from IPython.display import display\n","results=[json.loads(path.read_text()) for path in (PROJECT/'outputs').glob('*/validation_results.json')]\n","results=[r for r in results if r.get('completed')]\n","results.sort(key=lambda r:(int(r['group'][3:5]) if r['group'].startswith('run') and r['group'][3:5].isdigit() else 0,r['configuration']['learning_rate']))\n","rows=[dict(run=r['group'],lr=r['configuration']['learning_rate'],max_epochs=r['configuration']['epochs'],best_epoch=r['best_epoch'],**r['metrics']['overall']) for r in results]\n","display(pd.DataFrame(rows))\n","for r in results:\n","    print(r['group'],r['configuration'].get('tuning_reason','Completed first-try experiment'))\n","    display(pd.DataFrame(r['metrics']['per_label']).T)\n","selection=PROJECT/'outputs/best_model/selection.json'\n","if selection.exists():\n","    print('Overall best:',json.loads(selection.read_text()))\n","    print('Model:',PROJECT/'outputs/best_model')\n"],"execution_count":null,"outputs":[]}]}''')
nbformat.validate(nbformat.from_dict(nb))
with (p/'finetune.ipynb').open('x') as handle:json.dump(nb,handle,indent=2)
print('CREATED finetune.ipynb; launching Runs 4-5 and Validation-guided adaptive runs. No first-try reruns.')
command=[sys.executable,'-u',str(p/'train.py'),'--configs',str(p/'configs/run04_lr3e-5_ep8.json'),str(p/'configs/run05_lr4e-5_ep8.json'),'--adaptive']
with subprocess.Popen(command,env=dict(os.environ,PYTHONDONTWRITEBYTECODE='1'),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
    for line in process.stdout:print(line,end='',flush=True)
    assert process.wait()==0,'Tuning process stopped; inspect the error above.'


In [ ]:
from pathlib import Path
import json, subprocess, sys, os
p=Path('/content/drive/MyDrive/Holokai_Project/week9/OpenMed-NER-PharmaDetect-SuperClinical-434M')
s=(p/'train.py').read_text()
a=s.index('def consolidate_best(');b=s.index('def export_overall(',a)
replacement="""def consolidate_best(project, hashes):
    results=completed_results(project,hashes)
    check(bool(results),'Existing completed results are required before adaptive tuning')
    destination=project/'outputs'/'best_model'
    if not destination.exists():
        winner=max(results,key=lambda r:r['metrics']['overall']['f1'])
        source=project/'outputs'/winner['group']/'best_model'
        if (source/'model.safetensors').is_file() and (source/'config.json').is_file():
            source.rename(destination)
        else:
            destination.mkdir()
            write_json(destination/'selection.json',dict(group=winner['group'],epoch=winner['best_epoch'],validation_micro_f1=winner['metrics']['overall']['f1'],configuration=winner['configuration'],data_sha256=hashes,checkpoint_available=False))
            print('Historical checkpoint weights are unavailable; preserving metrics and establishing a new best available checkpoint.',flush=True)
    selected=global_selection(project)
    check(selected is not None and selected['data_sha256']==hashes,'Overall checkpoint provenance mismatch')
    print('Incumbent/reference:',selected['group'],selected['validation_micro_f1'],flush=True)

"""
s=s[:a]+replacement+s[b:]
s=s.replace("    model.save_pretrained(pending,safe_serialization=True)","    selection['checkpoint_available']=True\n    selection['historical_best_validation_micro_f1']=max(r['metrics']['overall']['f1'] for r in completed_results(project))\n    model.save_pretrained(pending,safe_serialization=True)")
s=s.replace("if selected is None or better(score,selected['validation_micro_f1']):","if selected is None or not selected.get('checkpoint_available',True) or better(score,selected['validation_micro_f1']):")
s=s.replace("print('Saved overall best:'","print('Saved best available checkpoint:'")
compile(s,'train.py','exec');(p/'train.py').write_text(s)
command=[sys.executable,'-u',str(p/'train.py'),'--configs',str(p/'configs/run04_lr3e-5_ep8.json'),str(p/'configs/run05_lr4e-5_ep8.json'),'--adaptive']
with subprocess.Popen(command,env=dict(os.environ,PYTHONDONTWRITEBYTECODE='1'),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
    for line in process.stdout:print(line,end='',flush=True)
    assert process.wait()==0,'Tuning stopped; inspect the error above.'


In [ ]:
from pathlib import Path
import os, subprocess, sys
PROJECT = Path("/content/drive/MyDrive/Holokai_Project/week9/OpenMed-NER-PharmaDetect-SuperClinical-434M")
CONFIGS = [PROJECT/'configs'/'first_try'/f'{group}.json' for group in ('learning_rate_1e-5', 'learning_rate_2e-5', 'learning_rate_3e-5')]
ENV = dict(os.environ, PYTHONDONTWRITEBYTECODE='1')
subprocess.run([sys.executable, str(PROJECT/'evaluate.py'), '--self-test'], env=ENV, check=True)
subprocess.run([sys.executable, str(PROJECT/'train.py'), '--check-data', '--configs', *map(str, CONFIGS)], env=ENV, check=True)


## Training — manual opt-in

This runs learning_rate_1e-5, learning_rate_2e-5, and learning_rate_3e-5 sequentially. Existing experiment folders are protected from overwriting.

In [ ]:
RUN_TRAINING = True
if RUN_TRAINING:
    subprocess.run([sys.executable, str(PROJECT/'train.py'), '--configs', *map(str, CONFIGS)], env=ENV, check=True)
else:
    print('Training disabled. Set RUN_TRAINING=True when ready.')


In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    result = subprocess.run(
        [sys.executable, "-u", str(PROJECT / "train.py"),
         "--configs", *map(str, CONFIGS)],
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    print(result.stdout)
    print("Return code:", result.returncode)
else:
    print("Training disabled. Set RUN_TRAINING=True when ready.")

In [ ]:
import json
import pandas as pd
from IPython.display import display
rows = []
for group in ('learning_rate_1e-5', 'learning_rate_2e-5', 'learning_rate_3e-5'):
    path = PROJECT/'outputs'/group/'validation_results.json'
    if path.exists():
        result = json.loads(path.read_text())
        if result['completed']:
            rows.append(dict(group=group, epoch=result['best_epoch'], learning_rate=result['configuration']['learning_rate'], **result['metrics']['overall']))
            print(group, 'per-label Validation metrics')
            display(pd.DataFrame(result['metrics']['per_label']).T)
if rows:
    display(pd.DataFrame(rows))
    winner = max(rows, key=lambda row: row['f1'])
    print('Best group:', winner['group'])
    print('Model:', PROJECT/'outputs'/winner['group']/'best_model')
else:
    print('No completed training results yet.')


In [ ]:
from pathlib import Path
import json, sys
p=Path('/content/drive/MyDrive/Holokai_Project/week9/OpenMed-NER-PharmaDetect-SuperClinical-434M')
print('TRAIN SOURCE\n'+(p/'train.py').read_text())
print('EVALUATION SOURCE\n'+(p/'evaluate.py').read_text())
for f in sorted((p/'outputs').glob('*/validation_results.json')):
    r=json.loads(f.read_text());print('EXISTING',str(f),json.dumps(r))
print('ROOT FILES', [x.name for x in p.iterdir()])
print('CUDA CHECK')
import torch
print(sys.version, torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
